# Data-Driven Dataset Extraction
This notebook performs Exploratory Data Analysis (EDA) on the Twitter, Reddit, and OpenAssistant datasets to find the true, empirically sound keywords for sentiment and agent actions. It then extracts the true Offline RL dataset.

In [ ]:
import pandas as pd
import json
import glob
import os
import re
import csv
from collections import Counter

STOPWORDS = set(["i", "me", "my", "we", "our", "you", "your", "he", "him", "she", "her", "it", "they", "them", 
                 "what", "which", "who", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", 
                 "been", "have", "has", "had", "do", "does", "did", "a", "an", "the", "and", "but", "if", "or", 
                 "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", 
                 "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", 
                 "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", 
                 "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", 
                 "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", 
                 "just", "don", "should", "now", "just", "us", "im", "it's", "dont", "can't", "please", "like", "get", 
                 "would", "got", "one", "know", "need", "could", "also", "see", "make", "https", "com"])

WORD_REGEX = re.compile(r'\b[a-z]{3,}\b')

def get_top_words(texts, top_n=30):
    cnt = Counter()
    for text in texts:
        if isinstance(text, str):
            tokens = WORD_REGEX.findall(text.lower())
            cnt.update([w for w in tokens if w not in STOPWORDS])
    return cnt.most_common(top_n)

def extract_sentiment(text, angry_words, happy_words):
    text = str(text).lower()
    if any(w in text for w in angry_words): return 0 # Angry
    if any(w in text for w in happy_words): return 2 # Happy
    return 1 # Neutral

def extract_action(text, action_dict):
    text = str(text).lower()
    for act_idx, keywords in action_dict.items():
        if any(w in text for w in keywords): return act_idx
    return 1 # Default

def get_reward(next_sentiment, done):
    if not done: return -1.0
    if next_sentiment == 2: return 10.0
    if next_sentiment == 0: return -10.0
    return -2.0

def save_dataset(tuples, name):
    path = f"offline_dataset_{name}.csv"
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["state", "action", "reward", "next_state", "done"])
        writer.writerows(tuples)
    print(f"\n> Saved {len(tuples)} transitions to {path}!")


## 1. Twitter Dataset Analysis & Extraction\nWe extract the top words to define `tw_angry` and `tw_happy` appropriately.

In [ ]:
TWITTER_PATH = r"D:\SEM_6\RL\Project\my_local_twitter\twcs\twcs.csv"
df_tw = pd.read_csv(TWITTER_PATH, nrows=5000) # Sample 5k for fast EDA
in_txt = df_tw[df_tw['inbound'] == True]['text'].tolist()
outb = df_tw[df_tw['inbound'] == False]['text'].tolist()

print("TOP CUSTOMER WORDS (Twitter):", get_top_words(in_txt, 20))
print("TOP AGENT WORDS (Twitter):", get_top_words(outb, 20))

# Tailored Twitter Dictionaries based on Customer Service tweets
tw_angry = ['why', 'wtf', 'hate', 'fix', 'issue', 'problem', 'error', 'awful', 'terrible', 'bad', 'worst', 'charged', 'stuck']
tw_happy = ['thanks', 'thank', 'amazing', 'working', 'solved', 'great', 'appreciate', 'perfect', 'love', 'done']

tw_actions = {
    2: ['sorry', 'apologies', 'apologize', 'frustrating', 'unfortunately'], # Affective repair
    0: ['dm', 'message', 'send', 'provide', 'details', 'account'], # Ask info
    3: ['email', 'call', 'team', 'form', 'specialist'], # Escalate
    5: ['update', 'investigating', 'looking', 'check'], # Proactive update
    6: ['expected', 'soon', 'patience', 'wait'], # Set Expectation
    4: ['let us know', 'help', 'anything else', 'reach out'] # Close feedback
}

print("\nProcessing full Twitter extraction...")
tuples_tw = []
df_tw_full = pd.read_csv(TWITTER_PATH, nrows=50000)
outbound = df_tw_full[df_tw_full['inbound'] == False].dropna(subset=['in_response_to_tweet_id'])
outbound_dict = {row['in_response_to_tweet_id']: row for _, row in outbound.iterrows()}
inbound_dict = {row['in_response_to_tweet_id']: row for _, row in df_tw_full.iterrows()}

for _, initial_tweet in df_tw_full[df_tw_full['inbound'] == True].iterrows():
    tid = initial_tweet['tweet_id']
    if tid in outbound_dict:
        agent_reply = outbound_dict[tid]
        agent_tid = agent_reply['tweet_id']
        state = extract_sentiment(initial_tweet['text'], tw_angry, tw_happy)
        action = extract_action(agent_reply['text'], tw_actions)
        if agent_tid in inbound_dict:
            customer_reply = inbound_dict[agent_tid]
            next_state = extract_sentiment(customer_reply['text'], tw_angry, tw_happy)
            tuples_tw.append((state, action, get_reward(next_state, True), next_state, True))

save_dataset(tuples_tw, "twitter")


## 2. Reddit Dataset Analysis & Extraction\nReddit dialogues might focus more on technical issues or forums.

In [ ]:
REDDIT_PATH = r"D:\SEM_6\RL\Project\my_local_dataset\training\training\*.txt"
files = glob.glob(REDDIT_PATH)
customer_texts_rd = []
agent_texts_rd = []

print("Sampling 100 Reddit threads for EDA...")
for fpath in files[:100]:
    with open(fpath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            turns = json.loads(line).get('turns', [])
            for i in range(len(turns)):
                if i % 2 == 0: customer_texts_rd.append(turns[i])
                else: agent_texts_rd.append(turns[i])

print("TOP CUSTOMER WORDS (Reddit):", get_top_words(customer_texts_rd, 20))
print("TOP AGENT WORDS (Reddit):", get_top_words(agent_texts_rd, 20))

# Tailored Reddit Dictionaries
rd_angry = ['issue', 'problem', 'stuck', 'error', 'failed', 'broken', 'wrong', 'bug', 'crash']
rd_happy = ['thanks', 'thank', 'worked', 'solved', 'great', 'awesome', 'cool']

rd_actions = {
    2: ['sorry', 'apologize', 'understand', 'my bad'],
    0: ['how', 'what', 'provide', 'details', 'log', 'screenshot', 'info'],
    3: ['email', 'contact', 'support', 'staff', 'admin'],
    5: ['update', 'check', 'investigate', 'testing'],
    6: ['soon', 'wait', 'next release'],
    4: ['let me know', 'anything else']
}

print("\nProcessing Reddit extraction...")
tuples_rd = []
for fpath in files[:500]: # Build dataset
    with open(fpath, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            turns = json.loads(line).get('turns', [])
            for i in range(0, len(turns) - 2, 2):
                state = extract_sentiment(turns[i], rd_angry, rd_happy)
                action = extract_action(turns[i+1], rd_actions)
                next_state = extract_sentiment(turns[i+2], rd_angry, rd_happy)
                done = (i + 2 >= len(turns) - 1 or len(turns) == i + 3)
                tuples_rd.append((state, action, get_reward(next_state, done), next_state, done))

save_dataset(tuples_rd, "reddit")


## 3. OASST Dataset Analysis & Extraction\nOpen Assistant typically involves instructions and cooperative AI interactions.

In [ ]:
OASST_PATH = r"D:\SEM_6\RL\Project\my_local_oasst1\2023-04-12_oasst_ready.trees.jsonl\2023-04-12_oasst_ready.trees.jsonl"
prompter_texts, assistant_texts = [], []

def recurse_tree(node):
    if not isinstance(node, dict): return
    text = str(node.get('text', ''))
    if node.get('role') == 'prompter': prompter_texts.append(text)
    elif node.get('role') == 'assistant': assistant_texts.append(text)
    for reply in node.get('replies', []): recurse_tree(reply)

if os.path.exists(OASST_PATH):
    print("Sampling OpenAssistant data for EDA...")
    with open(OASST_PATH, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i > 500: break # EDA sample
            if not line.strip(): continue
            recurse_tree(json.loads(line).get('prompt', {}))
    print("TOP CUSTOMER WORDS (OASST):", get_top_words(prompter_texts, 20))
    print("TOP AGENT WORDS (OASST):", get_top_words(assistant_texts, 20))

# Custom OASST Rules
oa_angry = ['wrong', 'incorrect', 'bad', 'error', 'failed', 'false', 'unhelpful']
oa_happy = ['thanks', 'thank', 'helpful', 'great', 'awesome', 'good', 'correct', 'agree']
oa_actions = {
    2: ['sorry', 'apologize', 'mistake', 'inaccurate'],
    0: ['clarify', 'provide', 'details', 'elaborate', 'context'],
    3: ['human', 'expert', 'developer'],
    5: ['update', 'check'],
    6: ['expected', 'guidelines'],
    4: ['help', 'anything else', 'further']
}

print("\nProcessing OASST extraction...")
tuples_oa = []
def extract_paths(node, cp=None):
    if cp is None: cp = []
    p = cp + [node]
    if 'replies' not in node or not node['replies']: return [p]
    return sum([extract_paths(r, p) for r in node['replies']], [])

if os.path.exists(OASST_PATH):
    with open(OASST_PATH, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i > 2500: break # Keep it reasonably sized
            if not line.strip(): continue
            paths = extract_paths(json.loads(line).get('prompt', {}))
            for path in paths:
                for j in range(len(path) - 2):
                    if path[j]['role'] == 'prompter' and path[j+1]['role'] == 'assistant' and path[j+2]['role'] == 'prompter':
                        state = extract_sentiment(path[j]['text'], oa_angry, oa_happy)
                        action = extract_action(path[j+1]['text'], oa_actions)
                        next_state = extract_sentiment(path[j+2]['text'], oa_angry, oa_happy)
                        done = (j + 2 >= len(path) - 1)
                        tuples_oa.append((state, action, get_reward(next_state, done), next_state, done))

save_dataset(tuples_oa, "oasst")
